In [3]:
import h5py
import matplotlib
from matplotlib import pyplot as plt
from berg import BERG
import numpy as np
import os
from PIL import Image
import torchvision
from torchvision import transforms as trn
from tqdm import tqdm
from IPython.display import display, JSON

berg_dir = "/Volumes/Extreme SSD/brain-encoding-response-generator" 
test_images = "Additionals/testimages/fmri_testimages"



images_dir = test_images
images_list = os.listdir(images_dir)
images_list.sort()

images = []
for img in tqdm(images_list):
    img_dir = os.path.join(images_dir, img)
    img = Image.open(img_dir).convert('RGB')
    # Center crop the images to square format, and resize them
    transform = trn.Compose([
        trn.CenterCrop(min(img.size)),
        trn.Resize((227,227))
    ])
    img = transform(img)
    img = np.asarray(img)
    img = img.transpose(2,0,1)
    images.append(img)
images = np.asarray(images)

# Print the images dimensions
print('\n\nImages shape:')
print(images.shape)
print('(Batch size × 3 RGB Channels x Width x Height)')


# Initialize the BERG object with the path to the toolkit directory
berg = BERG(berg_dir)

/Users/domenicbersch/anaconda3/envs/BERG/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
pixdim[1,2,3] should be non-zero; setting 0 dims to 1
100%|██████████| 100/100 [00:00<00:00, 197.11it/s]



Images shape:
(100, 3, 227, 227)
(Batch size × 3 RGB Channels x Width x Height)


### Load up all Models

In [2]:
# List all available models and their versions
available_models = berg.list_models()
print(f"Available models: {available_models}")

Available models: ['fmri-nsd-fwrf', 'fmri-nsd_fsaverage-vit_b_32', 'fmri-nsd_fsaverage-huze', 'fmri-things_fmri_1-vit_b_32', 'fmri-mosaic-CNN8_multihead_subNSD_verticesAll', 'fmri-mosaic-CNN8_multihead_subAll_verticesVisual', 'eeg-things_eeg_2-vit_b_32', 'meg-things_meg_1-vit_b_32', 'utah_array-tvsd-vit_b_32']


In [3]:
# Get a hierarchical view of available models by modality and dataset
catalog = berg.get_model_catalog(print_format=True)
print(f"Model Catalog as Dict: {catalog}")

Available Modalities and Datasets:
• EEG
  └─ THINGS EEG2

• MEG
  └─ THINGS MEG1

• UTAH ARRAYS
  └─ THINGS Ventral Stream Spiking Dataset (TVSD)

• FMRI
  └─ MOSAIC (NSD)
  └─ MOSAIC (all datasets)
  └─ Natural Scenes Dataset (NSD) (fsaverage surface space)
  └─ Natural Scenes Dataset (NSD) (subject-native volume space)
  └─ THINGS fMRI1

Model Catalog as Dict: {'fMRI': ['MOSAIC (NSD)', 'MOSAIC (all datasets)', 'Natural Scenes Dataset (NSD) (fsaverage surface space)', 'Natural Scenes Dataset (NSD) (subject-native volume space)', 'THINGS fMRI1'], 'EEG': ['THINGS EEG2'], 'MEG': ['THINGS MEG1'], 'Utah arrays': ['THINGS Ventral Stream Spiking Dataset (TVSD)']}


# fMRI Model

In [25]:
model_id = 'fmri-mosaic-CNN8_multihead_subNSD_verticesAll'
subject = 1

model_info = berg.describe(model_id)
print(list(model_info.keys()))

🧠 Model: fmri-mosaic-CNN8_multihead_subNSD_verticesAll

Modality: fMRI
Training dataset: MOSAIC (NSD)
Creator: MOSAIC Team (Lahner et al., 2025)

📋 Description:
This encoding model consists of a brain-optimized convolutional neural network
(CNN8) trained to predict whole-brain fMRI responses for Natural Scenes Dataset
(NSD) subjects. The model uses a shared 8-layer  convolutional core with
subject-specific linear factorized readout heads. Unlike the visual cortex
model, this variant predicts responses across 57,051 cortical vertices
(GlasserGroups 1-22) but is only trained on NSD subjects.  **Neural data.** The
model was trained on 8 subjects from the Natural Scenes Dataset (NSD). All data
underwent the shared MOSAIC preprocessing pipeline (fMRIPrep and GLMsingle) to
ensure consistency.  **Model architecture.** The CNN8 core consists of eight 2D
convolutional blocks (each with 2D convolution, batch normalization, and ReLU
activation). Convolutional kernel sizes range from 5×5 (blocks 1

### Normal Encoding

In [29]:
model_full = berg.get_encoding_model(model_id, 
                                     subject=subject, 
                                     device="auto")
pred_full = berg.encode(model_full, images)


metadata = berg.get_model_metadata(model_id, subject=subject)

print(metadata.keys())
for dataset, subjects in pred_full.items():
    for subject_data, data in subjects.items():
        print(dataset, "-", subject_data, data.shape)

Using cached checkpoint: ./mosaic_models/model-CNN8_framework-multihead_subjects-NSD_vertices-all.pth


/Users/domenicbersch/anaconda3/envs/BERG/lib/python3.9/site-packages/mosaic/models/readout.py:117: UserWarning: Readout is NOT initialized with mean activity but with 0!
  warnings.warn("Readout is NOT initialized with mean activity but with 0!")


MOSAIC model loaded on cpu for subject(s) [1]


Running batch inference on cpu: 100%|██████████| 4/4 [00:45<00:00, 11.41s/it]

dict_keys(['fmri', 'encoding_models'])
NaturalScenesDataset - sub-01 (100, 57051)


### Testing single ROI

In [7]:
model_v1 = berg.get_encoding_model(model_id, 
                                   subject=subject, 
                                   selection={"roi": ["L_V1"]}, 
                                   device="auto")

pred_v1 = berg.encode(model_v1, images)

for dataset, subjects in pred_v1.items():
    for subject_data, data in subjects.items():
        print(dataset, "-", subject_data, data.shape)
        
        
print("metadata: ", metadata["fmri"]["roi"]["L_V1"].shape)


Using cached checkpoint: ./mosaic_models/model-CNN8_framework-multihead_subjects-NSD_vertices-all.pth


/Users/domenicbersch/anaconda3/envs/BERG/lib/python3.9/site-packages/mosaic/models/readout.py:117: UserWarning: Readout is NOT initialized with mean activity but with 0!
  warnings.warn("Readout is NOT initialized with mean activity but with 0!")


MOSAIC model loaded on cpu for subject(s) [1]


Running batch inference on cpu: 100%|██████████| 4/4 [00:44<00:00, 11.12s/it]

NaturalScenesDataset - sub-01 (100, 816)
metadata:  (816,)


### Does the slicing work correctly?

In [30]:
subject = 1

# Model Full
model_full = berg.get_encoding_model(
    model_id, 
    subject=subject, 
    device="auto"
)
pred_full = berg.encode(model_full, images)
metadata = berg.get_model_metadata(model_id, subject=subject)


# Model ROI
roi_to_test = "L_V1"
model_roi = berg.get_encoding_model(
    model_id, 
    subject=subject, 
    selection={"roi": [roi_to_test]}, 
    device="auto"
)
pred_roi_direct = berg.encode(model_roi, images)


# Print our data shapes
pred_full_array = pred_full['NaturalScenesDataset'][f'sub-{subject:02d}']  # (57051,)
print(f"Full predictions shape: {pred_full_array.shape}")

pred_roi_direct_array = pred_roi_direct['NaturalScenesDataset'][f'sub-{subject:02d}']  # (816,)
print(f"Roi predictions shape: {pred_roi_direct_array.shape}")  

vertex_mapping = metadata["encoding_models"]["vertex_mapping_all"]  # (57051,)
print(f"Vertex mapping shape: {vertex_mapping.shape}")

roi_indices = metadata["fmri"]["roi"][roi_to_test]  # (816,)
print(f"ROI '{roi_to_test}' has {len(roi_indices)} vertices")


# Expand predictions to 91k space
pred_91k = np.full((pred_full_array.shape[0], 91282), np.nan)
pred_91k[:, vertex_mapping] = pred_full_array

# Slice with ROI indices
pred_roi_manual = pred_91k[:, roi_indices]

# Extract direct ROI result
pred_roi_direct_array = pred_roi_direct['NaturalScenesDataset'][f'sub-{subject:02d}']

# Compare
print(f"\nDirect shape: {pred_roi_direct_array.shape}")
print(f"Manual shape: {pred_roi_manual.shape}")
print(f"Match: {np.allclose(pred_roi_direct_array, pred_roi_manual, rtol=1e-5)}")

Using cached checkpoint: ./mosaic_models/model-CNN8_framework-multihead_subjects-NSD_vertices-all.pth


/Users/domenicbersch/anaconda3/envs/BERG/lib/python3.9/site-packages/mosaic/models/readout.py:117: UserWarning: Readout is NOT initialized with mean activity but with 0!
  warnings.warn("Readout is NOT initialized with mean activity but with 0!")


MOSAIC model loaded on cpu for subject(s) [1]


Running batch inference on cpu: 100%|██████████| 4/4 [00:45<00:00, 11.46s/it]


Using cached checkpoint: ./mosaic_models/model-CNN8_framework-multihead_subjects-NSD_vertices-all.pth


/Users/domenicbersch/anaconda3/envs/BERG/lib/python3.9/site-packages/mosaic/models/readout.py:117: UserWarning: Readout is NOT initialized with mean activity but with 0!
  warnings.warn("Readout is NOT initialized with mean activity but with 0!")


MOSAIC model loaded on cpu for subject(s) [1]


Running batch inference on cpu: 100%|██████████| 4/4 [00:45<00:00, 11.44s/it]

Full predictions shape: (100, 57051)
Roi predictions shape: (100, 816)
Vertex mapping shape: (57051,)
ROI 'L_V1' has 816 vertices

Direct shape: (100, 816)
Manual shape: (100, 816)
Match: True


### Testing two ROIs

In [9]:
model_v1v2 = berg.get_encoding_model(model_id, 
                                     subject=subject, 
                                     selection={"roi": ['L_V1', 'R_V1']}, 
                                     device="auto")
pred_v1v2 = berg.encode(model_v1v2, images)


for dataset, subjects in pred_v1v2.items():
    for subject_data, data in subjects.items():
        print(dataset, "-", subject_data, data.shape)

print("metadata: ", len(metadata["fmri"]["roi"]["L_V1"]) + len(metadata["fmri"]["roi"]["R_V1"]))


Using cached checkpoint: ./mosaic_models/model-CNN8_framework-multihead_subjects-NSD_vertices-all.pth


/Users/domenicbersch/anaconda3/envs/BERG/lib/python3.9/site-packages/mosaic/models/readout.py:117: UserWarning: Readout is NOT initialized with mean activity but with 0!
  warnings.warn("Readout is NOT initialized with mean activity but with 0!")


MOSAIC model loaded on cpu for subject(s) [1]


Running batch inference on cpu: 100%|██████████| 4/4 [00:44<00:00, 11.15s/it]

NaturalScenesDataset - sub-01 (100, 1603)
metadata:  1603


### Testing last 10 voxels

In [10]:
voxel_mask = np.zeros(57051, dtype=int)

voxel_mask[-10:] = 1
model_voxel = berg.get_encoding_model(model_id, 
                                      subject=subject, 
                                      selection={"voxel_index": voxel_mask}, 
                                      device="auto")
pred_voxel = berg.encode(model_voxel, images)



for dataset, subjects in pred_voxel.items():
    for subject_data, data in subjects.items():
        print(dataset, "-", subject_data, data.shape)


Using cached checkpoint: ./mosaic_models/model-CNN8_framework-multihead_subjects-NSD_vertices-all.pth


/Users/domenicbersch/anaconda3/envs/BERG/lib/python3.9/site-packages/mosaic/models/readout.py:117: UserWarning: Readout is NOT initialized with mean activity but with 0!
  warnings.warn("Readout is NOT initialized with mean activity but with 0!")


MOSAIC model loaded on cpu for subject(s) [1]


Running batch inference on cpu: 100%|██████████| 4/4 [00:44<00:00, 11.22s/it]

NaturalScenesDataset - sub-01 (100, 10)


### Testing combination

In [11]:
model_combined = berg.get_encoding_model(model_id, 
                                         subject=subject, 
                                         selection={"roi": ["L_V1"], "voxel_index": voxel_mask}, 
                                         device="auto")
pred_combined = berg.encode(model_combined, images)

for dataset, subjects in pred_combined.items():
    for subject_data, data in subjects.items():
        print(dataset, "-", subject_data, data.shape)
        
        
print("metadata: ", len(metadata["fmri"]["roi"]["L_V1"]) + 10)

Using cached checkpoint: ./mosaic_models/model-CNN8_framework-multihead_subjects-NSD_vertices-all.pth


/Users/domenicbersch/anaconda3/envs/BERG/lib/python3.9/site-packages/mosaic/models/readout.py:117: UserWarning: Readout is NOT initialized with mean activity but with 0!
  warnings.warn("Readout is NOT initialized with mean activity but with 0!")


MOSAIC model loaded on cpu for subject(s) [1]


Running batch inference on cpu: 100%|██████████| 4/4 [00:45<00:00, 11.25s/it]

NaturalScenesDataset - sub-01 (100, 826)
metadata:  826


### Testing list of subjects

In [ ]:

subject = [1,2]

model_combined = berg.get_encoding_model(model_id, 
                                         subject=subject, 
                                         selection={"roi": ["L_V1"], "voxel_index": voxel_mask}, 
                                         device="auto")
pred_combined_multi_sub = berg.encode(model_combined, images)

for dataset, subjects in pred_combined_multi_sub.items():
    for subject_data, data in subjects.items():
        print(dataset, "-", subject_data, data.shape)

Using cached checkpoint: ./mosaic_models/model-CNN8_framework-multihead_subjects-NSD_vertices-all.pth


/Users/domenicbersch/anaconda3/envs/BERG/lib/python3.9/site-packages/mosaic/models/readout.py:117: UserWarning: Readout is NOT initialized with mean activity but with 0!
  warnings.warn("Readout is NOT initialized with mean activity but with 0!")


MOSAIC model loaded on cpu for subject(s) [1, 2]


Running batch inference on cpu: 100%|██████████| 4/4 [00:54<00:00, 13.55s/it]

NaturalScenesDataset - sub-01 (100, 826)
NaturalScenesDataset - sub-02 (100, 826)
metadata:  1642


### Testing all subjects

In [ ]:
subject = "all"

model_combined = berg.get_encoding_model(model_id, 
                                         subject=subject, 
                                         selection={"roi": ["L_V1"], "voxel_index": voxel_mask}, 
                                         device="auto")
pred_all = berg.encode(model_combined, images)

for dataset, subjects in pred_all.items():
    for subject_data, data in subjects.items():
        print(dataset, "-", subject_data, data.shape)


Using cached checkpoint: ./mosaic_models/model-CNN8_framework-multihead_subjects-NSD_vertices-all.pth


/Users/domenicbersch/anaconda3/envs/BERG/lib/python3.9/site-packages/mosaic/models/readout.py:117: UserWarning: Readout is NOT initialized with mean activity but with 0!
  warnings.warn("Readout is NOT initialized with mean activity but with 0!")


MOSAIC model loaded on cpu for all subjects


Running batch inference on cpu: 100%|██████████| 4/4 [01:50<00:00, 27.52s/it]

NaturalScenesDataset - sub-01 (100, 826)
NaturalScenesDataset - sub-02 (100, 826)
NaturalScenesDataset - sub-03 (100, 826)
NaturalScenesDataset - sub-04 (100, 826)
NaturalScenesDataset - sub-05 (100, 826)
NaturalScenesDataset - sub-06 (100, 826)
NaturalScenesDataset - sub-07 (100, 826)
NaturalScenesDataset - sub-08 (100, 826)
metadata:  6538


# fMRI Model fmri-mosaic-CNN8_multihead_subAll_verticesVisual

In [4]:
model_id = 'fmri-mosaic-CNN8_multihead_subAll_verticesVisual'

subject = 'NSD-03'

In [32]:
model_full = berg.get_encoding_model(model_id, 
                                     subject=subject, 
                                     device="auto")
pred_full = berg.encode(model_full, images)


metadata = berg.get_model_metadata(model_id, subject=subject)

print(metadata.keys())

for dataset, subjects in pred_full.items():
    for subject_data, data in subjects.items():
        print(dataset, "-", subject_data, data.shape)

Using cached checkpoint: ./mosaic_models/model-CNN8_framework-multihead_subjects-all_vertices-visual.pth


/Users/domenicbersch/anaconda3/envs/BERG/lib/python3.9/site-packages/mosaic/models/readout.py:117: UserWarning: Readout is NOT initialized with mean activity but with 0!
  warnings.warn("Readout is NOT initialized with mean activity but with 0!")


MOSAIC model loaded on cpu for 1 subject(s)


Running batch inference on cpu: 100%|██████████| 4/4 [00:37<00:00,  9.40s/it]

dict_keys(['NaturalScenesDataset'])
NaturalScenesDataset - sub-03 (100, 7831)


### Does slicing work correclty?

In [12]:
metadata['NaturalScenesDataset']['sub-03'].keys()

dict_keys(['fmri', 'encoding_models'])

In [13]:
subject = 'NSD-03'

# Model Full
model_full = berg.get_encoding_model(
    model_id, 
    subject=subject, 
    device="auto"
)
pred_full = berg.encode(model_full, images)
metadata = berg.get_model_metadata(model_id, subject=subject)


# Model ROI
roi_to_test = "L_V1"
model_roi = berg.get_encoding_model(
    model_id, 
    subject=subject, 
    selection={"roi": [roi_to_test]}, 
    device="auto"
)
pred_roi_direct = berg.encode(model_roi, images)


# Print our data shapes
pred_full_array = pred_full['NaturalScenesDataset'][f'sub-03']  # (7831,)
print(f"Full predictions shape: {pred_full_array.shape}")

pred_roi_direct_array = pred_roi_direct['NaturalScenesDataset'][f'sub-03']  # (816,)
print(f"Roi predictions shape: {pred_roi_direct_array.shape}")  

vertex_mapping = metadata['NaturalScenesDataset']['sub-03']["encoding_models"]["vertex_mapping_visual"]  # (7831,)
print(f"Vertex mapping shape: {vertex_mapping.shape}")

roi_indices = metadata['NaturalScenesDataset']['sub-03']["fmri"]["roi"][roi_to_test]  # (816,)
print(f"ROI '{roi_to_test}' has {len(roi_indices)} vertices")


# Expand predictions to 91k space
pred_91k = np.full((pred_full_array.shape[0], 91282), np.nan)
pred_91k[:, vertex_mapping] = pred_full_array

# Slice with ROI indices
pred_roi_manual = pred_91k[:, roi_indices]

# Extract direct ROI result
pred_roi_direct_array = pred_roi_direct['NaturalScenesDataset'][f'sub-03']

# Compare
print(f"\nDirect shape: {pred_roi_direct_array.shape}")
print(f"Manual shape: {pred_roi_manual.shape}")
print(f"Match: {np.allclose(pred_roi_direct_array, pred_roi_manual, rtol=1e-5)}")

Using cached checkpoint: ./mosaic_models/model-CNN8_framework-multihead_subjects-all_vertices-visual.pth


/Users/domenicbersch/anaconda3/envs/BERG/lib/python3.9/site-packages/mosaic/models/readout.py:117: UserWarning: Readout is NOT initialized with mean activity but with 0!
  warnings.warn("Readout is NOT initialized with mean activity but with 0!")


MOSAIC model loaded on cpu for 1 subject(s)


Running batch inference on cpu: 100%|██████████| 4/4 [00:37<00:00,  9.26s/it]


Using cached checkpoint: ./mosaic_models/model-CNN8_framework-multihead_subjects-all_vertices-visual.pth


/Users/domenicbersch/anaconda3/envs/BERG/lib/python3.9/site-packages/mosaic/models/readout.py:117: UserWarning: Readout is NOT initialized with mean activity but with 0!
  warnings.warn("Readout is NOT initialized with mean activity but with 0!")


MOSAIC model loaded on cpu for 1 subject(s)


Running batch inference on cpu: 100%|██████████| 4/4 [00:37<00:00,  9.47s/it]

Full predictions shape: (100, 7831)
Roi predictions shape: (100, 816)
Vertex mapping shape: (7831,)
ROI 'L_V1' has 816 vertices

Direct shape: (100, 816)
Manual shape: (100, 816)
Match: True


### Testing single ROIs

In [16]:
model_v1 = berg.get_encoding_model(model_id, 
                                   subject=subject, 
                                   selection={"roi": ["L_V1"]}, 
                                   device="auto")

pred_v1 = berg.encode(model_v1, images)

for dataset, subjects in pred_v1.items():
    for subject_data, data in subjects.items():
        print(dataset, "-", subject_data, data.shape)
        
        
print("metadata: ", metadata['NaturalScenesDataset']['sub-03']["fmri"]["roi"]["L_V1"].shape)


Using cached checkpoint: ./mosaic_models/model-CNN8_framework-multihead_subjects-all_vertices-visual.pth


/Users/domenicbersch/anaconda3/envs/BERG/lib/python3.9/site-packages/mosaic/models/readout.py:117: UserWarning: Readout is NOT initialized with mean activity but with 0!
  warnings.warn("Readout is NOT initialized with mean activity but with 0!")


MOSAIC model loaded on cpu for 1 subject(s)


Running batch inference on cpu: 100%|██████████| 4/4 [00:36<00:00,  9.20s/it]

NaturalScenesDataset - sub-03 (100, 816)
metadata:  (816,)


### Testing two ROIs

In [17]:
model_v1v2 = berg.get_encoding_model(model_id, 
                                     subject=subject, 
                                     selection={"roi": ['L_V1', 'R_V1']}, 
                                     device="auto")
pred_v1v2 = berg.encode(model_v1v2, images)


for dataset, subjects in pred_v1v2.items():
    for subject_data, data in subjects.items():
        print(dataset, "-", subject_data, data.shape)

print("metadata: ", len(metadata['NaturalScenesDataset']['sub-03']["fmri"]["roi"]["L_V1"]) + len(metadata['NaturalScenesDataset']['sub-03']["fmri"]["roi"]["R_V1"]))


Using cached checkpoint: ./mosaic_models/model-CNN8_framework-multihead_subjects-all_vertices-visual.pth


/Users/domenicbersch/anaconda3/envs/BERG/lib/python3.9/site-packages/mosaic/models/readout.py:117: UserWarning: Readout is NOT initialized with mean activity but with 0!
  warnings.warn("Readout is NOT initialized with mean activity but with 0!")


MOSAIC model loaded on cpu for 1 subject(s)


Running batch inference on cpu: 100%|██████████| 4/4 [00:35<00:00,  8.96s/it]

NaturalScenesDataset - sub-03 (100, 1603)
metadata:  1603


### Testing last 10 voxels

In [18]:
voxel_mask = np.zeros(7831, dtype=int)

voxel_mask[-10:] = 1
model_voxel = berg.get_encoding_model(model_id, 
                                      subject=subject, 
                                      selection={"voxel_index": voxel_mask}, 
                                      device="auto")
pred_voxel = berg.encode(model_voxel, images)



for dataset, subjects in pred_voxel.items():
    for subject_data, data in subjects.items():
        print(dataset, "-", subject_data, data.shape)


Using cached checkpoint: ./mosaic_models/model-CNN8_framework-multihead_subjects-all_vertices-visual.pth


/Users/domenicbersch/anaconda3/envs/BERG/lib/python3.9/site-packages/mosaic/models/readout.py:117: UserWarning: Readout is NOT initialized with mean activity but with 0!
  warnings.warn("Readout is NOT initialized with mean activity but with 0!")


MOSAIC model loaded on cpu for 1 subject(s)


Running batch inference on cpu: 100%|██████████| 4/4 [00:36<00:00,  9.17s/it]

NaturalScenesDataset - sub-03 (100, 10)


### Testing combination 

In [19]:
model_combined = berg.get_encoding_model(model_id, 
                                         subject=subject, 
                                         selection={"roi": ["L_V1"], "voxel_index": voxel_mask}, 
                                         device="auto")
pred_combined = berg.encode(model_combined, images)

for dataset, subjects in pred_combined.items():
    for subject_data, data in subjects.items():
        print(dataset, "-", subject_data, data.shape)
        
        
print("metadata: ", len(metadata['NaturalScenesDataset']['sub-03']["fmri"]["roi"]["L_V1"]) + 10)

Using cached checkpoint: ./mosaic_models/model-CNN8_framework-multihead_subjects-all_vertices-visual.pth


/Users/domenicbersch/anaconda3/envs/BERG/lib/python3.9/site-packages/mosaic/models/readout.py:117: UserWarning: Readout is NOT initialized with mean activity but with 0!
  warnings.warn("Readout is NOT initialized with mean activity but with 0!")


MOSAIC model loaded on cpu for 1 subject(s)


Running batch inference on cpu: 100%|██████████| 4/4 [00:36<00:00,  9.05s/it]

NaturalScenesDataset - sub-03 (100, 826)
metadata:  826


### Testing list of subjects

In [33]:

subject= ['THINGS-02', 'NSD-03']

model_combined = berg.get_encoding_model(model_id, 
                                         subject=subject, 
                                         selection={"roi": ["L_V1"], "voxel_index": voxel_mask}, 
                                         device="auto")
pred_combined_multi_sub = berg.encode(model_combined, images)

for dataset, subjects in pred_combined_multi_sub.items():
    for subject_data, data in subjects.items():
        print(dataset, "-", subject_data, data.shape)


Using cached checkpoint: ./mosaic_models/model-CNN8_framework-multihead_subjects-all_vertices-visual.pth


/Users/domenicbersch/anaconda3/envs/BERG/lib/python3.9/site-packages/mosaic/models/readout.py:117: UserWarning: Readout is NOT initialized with mean activity but with 0!
  warnings.warn("Readout is NOT initialized with mean activity but with 0!")


MOSAIC model loaded on cpu for 2 subject(s)


Running batch inference on cpu: 100%|██████████| 4/4 [00:37<00:00,  9.42s/it]

THINGS - sub-02 (100, 826)
NaturalScenesDataset - sub-03 (100, 826)
